In [10]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "AACpassword123!"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__, assets_folder='/code_files', assets_url_path='assets')

# Grazioso Salvare logo is loaded from the Dash assets folder.


app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('Project Two Dashboard'))),
    html.Center(html.H4('Daniel Cervantes | CS-340')),
    html.Center(html.P('Codio Username: dcervantes597017419')),
    html.Center(
        html.A(
            html.Img(
                src='data:image/png;base64,' + base64.b64encode(
                    open('Grazioso Salvare Logo.png', 'rb').read()
                ).decode('utf-8'),
                style={'height': '150px'}
            ),
            href='https://www.snhu.edu',
            target='_blank'
        )
    ),
    # Rescue type selection
    html.Div([
        html.Label('Select Rescue Type:'),
        dcc.RadioItems(
            id='rescue-type-filter',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                 {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster/Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset', 'value': 'reset'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'marginRight': '20px'}
        )
    ], style={'textAlign': 'center', 'padding': '20px'}),

        # Separate rescue filters from the animal data table.
    html.Hr(),
    
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                         page_size=10,
                         page_action='native',
                         sort_action='native',
                         filter_action='native',
                         row_selectable='single',
                         selected_rows=[0],
                         selected_columns=[],
                         style_table={'overflowX': 'auto'},
                         style_cell={
                             'textAlign': 'left',
                             'padding': '8px',
                             'minWidth': '100px',
                             'maxWidth': '180px',
                             'whiteSpace': 'normal'
                         },

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('rescue-type-filter', 'value')])
def update_dashboard(filter_type):

    if filter_type == 'water':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$in': [
                    'Labrador Retriever Mix',
                    'Chesapeake Bay Retriever',
                    'Newfoundland'
                ]
            },
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }

        filtered_df = pd.DataFrame.from_records(db.read(query))
        filtered_df.drop(columns=['_id'], errors='ignore', inplace=True)
        
        return filtered_df.to_dict('records')

    elif filter_type == 'mountain':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$in': [
                    'German Shepherd',
                    'Alaskan Malamute',
                    'Old English Sheepdog',
                    'Siberian Husky',
                    'Rottweiler'
                ]
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }

        filtered_df = pd.DataFrame.from_records(db.read(query))
        filtered_df.drop(columns=['_id'], errors='ignore', inplace=True)

        return filtered_df.to_dict('records')

    elif filter_type == 'disaster':
        query = {
            'animal_type': 'Dog',
            'breed': {
                '$in': [
                    'Doberman Pinscher',
                    'German Shepherd',
                    'Golden Retriever',
                    'Bloodhound',
                    'Rottweiler'
                ]
            },
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {
                '$gte': 20,
                '$lte': 300
            }
        }

        filtered_df = pd.DataFrame.from_records(db.read(query))
        filtered_df.drop(columns=['_id'], errors='ignore', inplace=True)

        return filtered_df.to_dict('records')

    elif filter_type == 'reset':
        filtered_df = pd.DataFrame.from_records(db.read({}))
        filtered_df.drop(columns=['_id'], errors='ignore', inplace=True)

        return filtered_df.to_dict('records')
    
    else:
        return df.to_dict('records')


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    if viewData is None:
        dff = df
    else:
        dff = pd.DataFrame.from_records(viewData)

    if dff.empty:
        return [html.P('No animals match the selected filter.')]
    
    # Show the 10 most common breeds and group the rest as Other.
    breed_counts = dff['breed'].value_counts()
    top_breeds = breed_counts.head(10).index

    dff = dff.copy()
    dff['breed'] = dff['breed'].where(
        dff['breed'].isin(top_breeds),
        'Other'
    )

    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names='breed',
                title='Animal Breeds by Rescue Category'
            )
        )
    ]

    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        selected_columns = []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    if not viewData:
        return [html.P('No animal locations available for this filter.')]

    if not index or index[0] >= len(viewData):
        index = [0]
    
        # Convert the currently displayed animal records into a DataFrame.
    dff = pd.DataFrame.from_records(viewData)

    # Use the selected animal's row index.
    row = index[0]
    
    # Check that the selected animal has valid map coordinates.
    latitude = pd.to_numeric(
        dff.iloc[row]['location_lat'], errors='coerce'
    )
    longitude = pd.to_numeric(
        dff.iloc[row]['location_long'], errors='coerce'
    )

    if not np.isfinite(latitude) or not np.isfinite(longitude):
        return [html.P('No valid location is available for this animal.')]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[float(latitude), float(longitude)], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Use the animal's latitude and longitude for the map marker.
            # Display the animal's breed and name when the marker is selected.
            dl.Marker(
            position=[float(latitude), float(longitude)],
            children=[
                dl.Tooltip(dff.iloc[row]['breed']),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row]['name'])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://brigadeurgent-obscureexport-3000.codio.io/proxy/8050/
